# Part 1: Data Processing (~1 page).

## Task 1: Retrieve sample:
Your first task is to retrieve a sample of the FakeNewsCorpus from https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv to an external site.
and structure, process, clean it. You should follow the methodology you developed in Exercise 1.

In [1]:
import pandas as pd

def getgit(url):
    data = pd.read_csv(
        url,
        encoding="utf-8",
        engine="python"  
    )
    return data

fnc = getgit("https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv")

print(fnc.head())
print(fnc.shape)


   Unnamed: 0   id                domain        type  \
0           0  141               awm.com  unreliable   
1           1  256     beforeitsnews.com        fake   
2           2  700           cnnnext.com  unreliable   
3           3  768               awm.com  unreliable   
4           4  791  bipartisanreport.com   clickbait   

                                                 url  \
0  http://awm.com/church-congregation-brings-gift...   
1  http://beforeitsnews.com/awakening-start-here/...   
2  http://www.cnnnext.com/video/18526/never-hike-...   
3  http://awm.com/elusive-alien-of-the-sea-caught...   
4  http://bipartisanreport.com/2018/01/21/trumps-...   

                                             content  \
0  Sometimes the power of Christmas will make you...   
1  AWAKENING OF 12 STRANDS of DNA – “Reconnecting...   
2  Never Hike Alone: A Friday the 13th Fan Film U...   
3  When a rare shark was caught, scientists were ...   
4  Donald Trump has the unnerving ability to a

### Define method to clean text:

In [2]:
#clean text
def clean_text(text):
    
    text = text.apply(lambda x: x.str.lower() if x.dtype == 'object' else x) # all words must be lowercased 
    text = text.apply(lambda x: x.str.strip() if x.dtype == 'object' else x) # it should not contain multiple white spaces, tabs, or new lines
    text = text.apply(lambda x: x.str.replace(r"\bhttps?://[^\s]+|www\.[^\s]+", "<URL>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", "<EMAIL>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"\b(\d{4}-\d{2}-\d{2}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b", "<DATE>", regex = True) if x.dtype == 'object' else x)
    text = text.apply(lambda x: x.str.replace(r"\d+", "<NUM>", regex = True) if x.dtype == 'object' else x)    
    
    return text

text = clean_text(fnc)

### Import required packages, and clean text:

In [3]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt_tab')
nltk.download('stopwords')

if isinstance(text, pd.DataFrame):
    one_string = "\n".join(text.astype(str).apply(lambda row: " ".join(row.dropna().tolist()), axis=1).tolist()
    )

elif isinstance(text, list):
    one_string = " ".join(str(x) for x in text if x is not None)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Kian\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kian\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Tokenize:
### What and Why it's used:
nltk is used to tokenzie as it's efficient and strongly hinted at by the task description.

In [4]:
tokens = word_tokenize(one_string)

token_count = len(tokens)
vocab_size = len(set(tokens))

print(f"Token count: {token_count}.")
print(f"Vocabulary size: {vocab_size}.")

Token count: 232913.
Vocabulary size: 18147.


### Remove Stopwords, and compute size:
### What and Why it's used:
nltk is used to remove stopwords as it's efficient and strongly hinted at by the task description.

In [5]:
stop_words = set(stopwords.words("english"))
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

#filtered_text = " ".join(filtered_tokens) 

vocab_nostop = len(set(filtered_tokens))

#compute size of vocab:
print(f"Size of vocabulary after removing stopwords: {vocab_nostop}.")

#Compute the reduction rate of the vocabulary size after removing stopwords:
print(f"Reduction in size after removing stopwords: {((vocab_size-vocab_nostop)/vocab_size)*100:.3f} %.")

Size of vocabulary after removing stopwords: 18001.
Reduction in size after removing stopwords: 0.805 %.


### Stemming and compute size of vocabolary:
### What and Why it's used:
nltk PorterStemmer, because why not.

In [6]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

stemmed_tokens = [ps.stem(word) for word in filtered_tokens]

vocab_stem = len(set(stemmed_tokens))

#Compute size of vocabulary:
print(f"Vocabulary size after stemming: {vocab_stem}.")

#Compute the reduction rate of the vocabulary size after stemming:
print(f"Reduction in size after removing word variations: {((vocab_nostop - vocab_stem) / vocab_nostop)*100:.3f} %.")

Vocabulary size after stemming: 12784.
Reduction in size after removing word variations: 28.982 %.


# Task 2 & 3: Pipeline and Observations

### Importing libraries and downloading the resources:

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

### Tokenize, Remove stopwords, Stemming:

In [ ]:
#Tokenization:
import re

def tokenize_fast(text):
    return re.findall(r"\b\w+\b", text.lower())

#Remove Stopwords:
def remove_stopwords(tokens):
    return [word for word in tokens if word.lower() not in stop_words]

#Remove Stemming:
def stem_tokens(tokens):
    return [stemmer.stem(word) for word in tokens]

In [ ]:
df = pd.read_csv("995,000_rows.csv", nrows=5)
print(df.columns)

### Process large dataset in chunks:

In [ ]:
chunk_size = 10000

vocab_original = set()
vocab_no_stop = set()
vocab_stemmed = set()

for chunk in pd.read_csv("995,000_rows.csv", chunksize=chunk_size):

    texts = chunk["content"].dropna()

    for text in texts:

        tokens = tokenize_fast(text)
        vocab_original.update(tokens)

        tokens_no_stop = remove_stopwords(tokens)
        vocab_no_stop.update(tokens_no_stop)

        stemmed = stem_tokens(tokens_no_stop)
        vocab_stemmed.update(stemmed)

### Saving data:

In [ ]:
import pickle

with open("vocab_original.pkl", "wb") as f:
    pickle.dump(vocab_original, f)

with open("vocab_no_stop.pkl", "wb") as f:
    pickle.dump(vocab_no_stop, f)

with open("vocab_stemmed.pkl", "wb") as f:
    pickle.dump(vocab_stemmed, f)

### If needing to load it later:

In [ ]:
import pickle

with open("vocab_original.pkl", "rb") as f:
    vocab_original = pickle.load(f)

with open("vocab_no_stop.pkl", "rb") as f:
    vocab_no_stop = pickle.load(f)

with open("vocab_stemmed.pkl", "rb") as f:
    vocab_stemmed = pickle.load(f)

### Save computed numbers:

In [ ]:
V0 = len(vocab_original)
V1 = len(vocab_no_stop)
V2 = len(vocab_stemmed)

stop_reduction = (V0 - V1) / V0
stem_reduction = (V1 - V2) / V1

In [ ]:
import json

results = {
    "vocab_original": V0,
    "vocab_no_stop": V1,
    "vocab_stemmed": V2,
    "stopword_reduction": stop_reduction,
    "stemming_reduction": stem_reduction
}

with open("preprocessing_results.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
import inspect
print(inspect.getsource(tokenize_text))

In [ ]:
import pickle

pickle.dump(vocab_original, open("vocab_original.pkl", "wb"))
pickle.dump(vocab_no_stop, open("vocab_no_stop.pkl", "wb"))
pickle.dump(vocab_stemmed, open("vocab_stemmed.pkl", "wb"))

### Altering code to collect frequencies:

In [ ]:
from collections import Counter
import pandas as pd
import re

word_freq = Counter()
url_count = 0
num_count = 0

chunk_size = 10000

for chunk in pd.read_csv("995,000_rows.csv", chunksize=chunk_size):

    texts = chunk["content"].dropna()

    for text in texts:

        tokens = tokenize_fast(text)

        word_freq.update(tokens)

        url_count += text.count("http")
        num_count += len(re.findall(r"\d+", text))

In [ ]:
#Compute most frequent words:
top_100 = word_freq.most_common(100)

for word, freq in top_100[:20]:
    print(word, freq)

In [ ]:
#Frequency Distribution:
import matplotlib.pyplot as plt

top_10000 = word_freq.most_common(10000)
freqs = [freq for word, freq in top_10000]

plt.figure(figsize=(10,6))
plt.plot(freqs)
plt.title("Word Frequency Distribution")
plt.xlabel("Word Rank")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#Frequency of URL and NUM:
print("Total URLs detected:", url_count)
print("Total numeric tokens:", num_count)

#Normalized:
rows = 995000

print("URLs per article:", url_count / rows)
print("Numbers per article:", num_count / rows)

In [ ]:
#How many words appear once:
hapax = sum(1 for w, c in word_freq.items() if c == 1)

print("Words appearing once:", hapax)
print("Percentage of rare words:", hapax / len(word_freq))

In [ ]:
#Saving frequencies:
import pickle

pickle.dump(word_freq, open("word_freq.pkl", "wb"))


# Task 4: Split database to training and validation

### Loading Data:

In [ ]:
import pandas as pd

df = pd.read_csv("995,000_rows.csv")

### Performing split using sklearn:

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=69
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=69
)

In [ ]:
#Control:
print(len(train_df))
print(len(val_df))
print(len(test_df))